In [1]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
import os
import gradio as gr
# 1. 使用官方DeepSeek集成包，而非ChatOpenAI
from langchain_deepseek import ChatDeepSeek  
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

# ==================== 配置区域（请按需修改） ====================
PDF_PATH = r"D:\360MoveData\Users\不遵\Desktop\xiaogui-副本.pdf"
PERSIST_DIR = "./faiss_index"
# 模型参数
MODEL_NAME = "deepseek-v4-pro" # 根据你的API有效模型填写
API_KEY = "sk-de92cfd6fecd4d1b89101b7d6fc8344b"
# API地址修正为官方标准地址
BASE_URL = "https://api.deepseek.com"  
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50
RETRIEVAL_K = 3
TEMPERATURE = 0

store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

# ==================== 1. 加载并切分文档 ====================
def load_and_split_document(pdf_path):
    if not os.path.exists(pdf_path):
        raise FileNotFoundError(f"PDF 文件不存在: {pdf_path}")
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", "。", "！", "？", "；", "，", " ", ""]
    )
    docs = text_splitter.split_documents(documents)
    print(f"✓ 文档加载完成，共切分为 {len(docs)} 个文本块")
    return docs

# ==================== 2. 构建或加载 FAISS 向量库 ====================
def get_vectorstore(docs, embeddings, persist_dir):
    if os.path.exists(persist_dir) and os.path.isdir(persist_dir):
        print(f"✓ 发现已有 FAISS 索引，从 {persist_dir} 加载")
        vectorstore = FAISS.load_local(persist_dir, embeddings, allow_dangerous_deserialization=True)
    else:
        print("✓ 正在创建 FAISS 向量数据库...")
        vectorstore = FAISS.from_documents(docs, embeddings)
        vectorstore.save_local(persist_dir)
        print(f"✓ FAISS 索引已保存至 {persist_dir}")
    return vectorstore

# ==================== 3. 辅助函数：格式化检索到的文档 ====================
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# ==================== 4. 构建 RAG 链（LCEL + 记忆） ====================
def build_rag_chain(vectorstore, api_key, model_name, base_url, temperature):
    llm = ChatDeepSeek(
        api_key=api_key,
        base_url=base_url,
        model=model_name,
        temperature=temperature,
        top_p=0.9,
        max_tokens=4096,
    )

    retriever = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": RETRIEVAL_K}
    )

    system_prompt = (
        "你是一个专门回答长春工业大学校规的问答助手，你的回答风格亲切自然。\n"
        "请严格依据以下“上下文”内容回答用户的问题。\n"
        "如果上下文信息不足以回答问题，请礼貌地说明你不知道，不要编造答案。\n\n"
        "上下文：\n{context}"
    )
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}")
    ])

    # 修正：定义一个可调用对象来检索并格式化文档
    def retrieve_and_format(inputs):
        docs = retriever.invoke(inputs["input"])
        return format_docs(docs)

    rag_chain = (
        RunnablePassthrough.assign(context=retrieve_and_format)
        | prompt
        | llm
        | StrOutputParser()
    )

    conversational_chain = RunnableWithMessageHistory(
        rag_chain,
        get_session_history,
        input_messages_key="input",
        history_messages_key="chat_history",
    )
    return conversational_chain

# ==================== 5. Gradio 聊天界面 ====================
def chat_interface(chain):
    def respond(message, history):
        session_id = "user_123"
        # 3. 确保输入键是 "input"
        response = chain.invoke(
            {"input": message},
            config={"configurable": {"session_id": session_id}}
        )
        return response

    demo = gr.ChatInterface(
        fn=respond,
        title="📚 长春工业大学校规问答模型 · 文档知识助手",
        description=(
            "基于 DeepSeek-v4-pro 模型 + FAISS 检索 + RAG 构建。\n"
            "支持多轮对话记忆，可针对上传的 PDF 文档内容提问。"
        ),
        examples=[
            ["这篇文档主要讲了什么？"],
            ["能总结一下第二章节的核心观点吗？"],
            ["文档中提到了哪些校规？"]
        ]
    )
    return demo

# ==================== 6. 主流程 ====================
def main():
    print("=" * 50)
    print("LangChain + FAISS + RAG 文档问答系统启动")
    print("=" * 50)

    print("\n[1/5] 加载并切分 PDF 文档...")
    try:
        docs = load_and_split_document(PDF_PATH)
    except Exception as e:
        print(f"❌ 文档加载失败: {e}")
        return

    print("\n[2/5] 初始化 Embedding 模型...")
    embeddings =HuggingFaceEmbeddings(
        model_name=r"D:\pp\models\bge-small-zh-v1.5",
        model_kwargs={'device': 'cpu'},
        encode_kwargs={'normalize_embeddings': True}
    )

    print("\n[3/5] 构建或加载向量数据库...")
    vectorstore = get_vectorstore(docs, embeddings, PERSIST_DIR)

    print("\n[4/5] 构建 RAG 对话链...")
    rag_chain = build_rag_chain(vectorstore, API_KEY, MODEL_NAME, BASE_URL, TEMPERATURE)

    print("\n[5/5] 启动 Web 界面...")
    demo = chat_interface(rag_chain)
    demo.launch(share=True, server_name="0.0.0.0", server_port=7860)

if __name__ == "__main__":
    main()

C:\Users\不遵\AppData\Local\Temp\ipykernel_27584\1622167909.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings



LangChain + FAISS + RAG 文档问答系统启动

[1/5] 加载并切分 PDF 文档...
✓ 文档加载完成，共切分为 426 个文本块

[2/5] 初始化 Embedding 模型...

[3/5] 构建或加载向量数据库...
✓ 发现已有 FAISS 索引，从 ./faiss_index 加载

[4/5] 构建 RAG 对话链...


C:\Users\不遵\AppData\Local\Temp\ipykernel_27584\1622167909.py:158: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings =HuggingFaceEmbeddings(
C:\Users\不遵\AppData\Local\Temp\ipykernel_27584\1622167909.py:168: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  rag_chain = build_rag_chain(vectorstore, API_KEY, MODEL_NAME, BASE_URL, TEMPERATURE)



[5/5] 启动 Web 界面...
* Running on local URL:  http://0.0.0.0:7860

Could not create share link. Missing file: D:\HuggingFaceCache\gradio\frpc\frpc_windows_amd64_v0.3. 

Please check your internet connection. This can happen if your antivirus software blocks the download of this file. You can install manually by following these steps: 

1. Download this file: https://cdn-media.huggingface.co/frpc-gradio-0.3/frpc_windows_amd64.exe
2. Rename the downloaded file to: frpc_windows_amd64_v0.3
3. Move the file to this location: D:\HuggingFaceCache\gradio\frpc
